# Yellow Taxi cleaning and validation
Run after the ingestion notebook, in the same Fabric Lakehouse. This notebook reads `bronze_trips` and the zone CSV, profiles the source, writes Silver, and creates the fact and dimension tables. Use a fresh set of output tables for a full run. The writes stop if an output table already exists. The findings below refer to the submitted 2022–2023 data.

In [ ]:
%%configure -f
{
  "conf": {
    "spark.native.enabled": "false",
    "spark.gluten.sql.native.writer.enabled": "false",
    "spark.sql.cbo.enabled": "false",
    "spark.sql.cbo.planStats.enabled": "false"
  }
}

## Check the taxi zone lookup

In [ ]:
# Check that each LocationID is present and unique.
from pyspark.sql import functions as F

zone_lookup_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/reference/taxi_zone_lookup.csv")
)

lookup_summary = zone_lookup_df.agg(
    F.count("*").alias("lookup_rows"),
    F.countDistinct("LocationID").alias("distinct_location_ids"),
    F.coalesce(
        F.sum(F.when(F.col("LocationID").isNull(), 1).otherwise(0)),
        F.lit(0)
    ).alias("null_location_ids")
).first()

duplicate_id_groups = (
    zone_lookup_df
    .groupBy("LocationID")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Lookup rows: {lookup_summary['lookup_rows']}")
print(f"Distinct LocationID values: {lookup_summary['distinct_location_ids']}")
print(f"Null LocationID values: {lookup_summary['null_location_ids']}")
print(f"Duplicate LocationID groups: {duplicate_id_groups}")

# Duplicate or empty lookup keys would make location counts unreliable.
if (not lookup_summary["lookup_rows"] or lookup_summary["null_location_ids"]
        or duplicate_id_groups):
    raise ValueError("The zone lookup must contain one non-null key per location.")


## Check trip data quality

In [ ]:
# Profile missing values and key trip quality issues.
from pyspark.sql import functions as F

bronze_df = spark.table("bronze_trips")

zone_lookup_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/reference/taxi_zone_lookup.csv")
)

zone_ids = zone_lookup_df.select(
    F.col("LocationID").cast("long").alias("LocationID")
).distinct()

# Broadcast sends the small zone list to each worker so it can match trips locally.
pickup_lookup = F.broadcast(
    zone_ids.select(F.col("LocationID").alias("pickup_lookup_id"))
)

dropoff_lookup = F.broadcast(
    zone_ids.select(F.col("LocationID").alias("dropoff_lookup_id"))
)

# The left joins keep trips with unmatched zone IDs. A missing lookup key can then be counted in the quality checks.
profile_df = (
    bronze_df.alias("t")
    .join(
        pickup_lookup,
        F.col("t.PULocationID") == F.col("pickup_lookup_id"),
        "left"
    )
    .join(
        dropoff_lookup,
        F.col("t.DOLocationID") == F.col("dropoff_lookup_id"),
        "left"
    )
)

# Each matching row contributes 1 and every other row contributes 0. Their sum is the number of trips with that issue.
def flag_count(condition, metric_name):
    return F.sum(
        F.when(condition, 1).otherwise(0)
    ).cast("long").alias(metric_name)

profile_expressions = [F.count("*").alias("profile_rows")]

for column_name in bronze_df.columns:
    profile_expressions.append(
        flag_count(
            F.col(f"t.{column_name}").isNull(),
            f"missing__{column_name}"
        )
    )

pickup_time = F.col("t.tpep_pickup_datetime")
dropoff_time = F.col("t.tpep_dropoff_datetime")
duration_minutes = (
    F.unix_timestamp(dropoff_time) -
    F.unix_timestamp(pickup_time)
) / 60.0

profile_expressions.extend([
    flag_count(
        F.col("t.trip_distance") <= 0,
        "trip_distance_zero_or_negative"
    ),
    flag_count(
        pickup_time.isNotNull()
        & dropoff_time.isNotNull()
        & (dropoff_time <= pickup_time),
        "duration_zero_or_negative"
    ),
    flag_count(
        duration_minutes > 240,
        "duration_over_4_hours"
    ),
    flag_count(
        duration_minutes > 1440,
        "duration_over_24_hours"
    ),
    flag_count(
        (F.col("t.passenger_count") <= 0),
        "passenger_count_zero_or_negative"
    ),
    flag_count(
        F.col("t.passenger_count") > 6,
        "passenger_count_over_6"
    ),
    flag_count(
        F.col("t.PULocationID").isNotNull()
        & F.col("pickup_lookup_id").isNull(),
        "unmatched_pickup_location_id"
    ),
    flag_count(
        F.col("t.DOLocationID").isNotNull()
        & F.col("dropoff_lookup_id").isNull(),
        "unmatched_dropoff_location_id"
    ),
    flag_count(
        pickup_time.isNotNull()
        & (
            (F.date_format(pickup_time, "yyyy-MM-dd") < "2022-01-01")
            | (F.date_format(pickup_time, "yyyy-MM-dd") >= "2024-01-01")
        ),
        "pickup_outside_study_period"
    ),
    flag_count(
        pickup_time.isNotNull()
        & (
            F.date_format(pickup_time, "yyyy-MM")
            != F.col("t.source_month")
        ),
        "pickup_month_differs_from_source_month"
    )
])

financial_columns = [
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "airport_fee"
]

for column_name in financial_columns:
    profile_expressions.append(
        flag_count(
            F.col(f"t.{column_name}") < 0,
            f"negative__{column_name}"
        )
    )

profile_values = profile_df.agg(*profile_expressions).first().asDict()
# Use the number of rows in this run for all percentages.
total_rows = profile_values.pop("profile_rows")
if total_rows == 0:
    raise ValueError("Bronze is empty. Check the ingestion notebook.")

for metric_name, metric_value in profile_values.items():
    value = int(metric_value or 0)
    percentage = value / total_rows * 100
    print(f"{metric_name}: {value:,} ({percentage:.4f}%)")

In [ ]:
# Check whether the five related fields are missing together.
from functools import reduce
from pyspark.sql import functions as F

bronze_df = spark.table("bronze_trips")

fields_to_check = [
    "passenger_count",
    "RatecodeID",
    "store_and_fwd_flag",
    "congestion_surcharge",
    "airport_fee"
]

null_conditions = [F.col(field).isNull() for field in fields_to_check]
# The | operator flags a row if any field is missing. The & operator below requires all five fields to be missing on that same row.
any_missing = reduce(lambda left, right: left | right, null_conditions)
all_five_missing = reduce(lambda left, right: left & right, null_conditions)

missing_by_month_vendor = (
    bronze_df
    .groupBy("source_month", "VendorID")
    .agg(
        F.count("*").alias("trip_rows"),
        F.sum(F.when(any_missing, 1).otherwise(0)).alias("rows_with_any_missing"),
        F.sum(F.when(all_five_missing, 1).otherwise(0)).alias("rows_with_all_five_missing"),
        *[
            F.sum(F.when(F.col(field).isNull(), 1).otherwise(0))
            .alias(f"missing_{field}")
            for field in fields_to_check
        ]
    )
    .filter(F.col("rows_with_any_missing") > 0)
    .orderBy("source_month", "VendorID")
    .collect()
)

for row in missing_by_month_vendor:
    print(
        f"{row['source_month']} | VendorID={row['VendorID']} "
        f"| rows={row['trip_rows']:,} "
        f"| any_missing={row['rows_with_any_missing']:,} "
        f"| all_five_missing={row['rows_with_all_five_missing']:,}"
    )   

In [ ]:
# Find exact duplicate trip groups and extra copies.
from pyspark.sql import functions as F

bronze_df = spark.table("bronze_trips")

# File names describe the source. Only trip attributes define a duplicate.
trip_columns = [
    column_name
    for column_name in bronze_df.columns
    if column_name not in {"source_file", "source_month"}
]

# Spark packs the trip fields into one record, converts it to JSON and hashes it so matching trips can be grouped.
row_fingerprints = bronze_df.select(
    F.sha2(
        F.to_json(F.struct(*[F.col(column_name) for column_name in trip_columns])),
        256
    ).alias("row_hash")
)

duplicate_summary = (
    row_fingerprints
    .groupBy("row_hash")
    .count()
    .filter(F.col("count") > 1)
    .agg(
        F.count("*").alias("duplicate_groups"),
        # A group of three matching rows has two extra copies. Subtract one per group before adding them up.
        F.sum(F.col("count") - 1).alias("extra_duplicate_rows")
    )
    .first()
)

duplicate_groups = int(duplicate_summary["duplicate_groups"] or 0)
extra_duplicate_rows = int(duplicate_summary["extra_duplicate_rows"] or 0)

print(f"Exact duplicate groups: {duplicate_groups:,}")
print(f"Extra duplicate rows: {extra_duplicate_rows:,}")

In [ ]:
# Show the trip details for the exact duplicate groups.
from pyspark.sql import functions as F

bronze_df = spark.table("bronze_trips")

trip_columns = [
    column_name
    for column_name in bronze_df.columns
    if column_name not in {"source_file", "source_month"}
]

bronze_with_hash = bronze_df.withColumn(
    "row_hash",
    F.sha2(
        F.to_json(F.struct(*[F.col(column_name) for column_name in trip_columns])),
        256
    )
)

duplicate_keys = (
    bronze_with_hash
    .groupBy("row_hash")
    .agg(F.count("*").alias("duplicate_count"))
    .filter(F.col("duplicate_count") > 1)
)

# Join the repeated hashes back to Bronze to see the original file names and trip details for those groups.
duplicate_details = (
    bronze_with_hash
    .join(F.broadcast(duplicate_keys), "row_hash", "inner")
    .select(
        "row_hash",
        "duplicate_count",
        "source_month",
        "source_file",
        "VendorID",
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "trip_distance",
        "fare_amount",
        "total_amount"
    )
    .orderBy("row_hash", "source_month", "source_file")
)

duplicate_details.show(20, truncate=False)

### Duplicate trips
The exact-duplicate review identified 3 groups and 3 excess rows. All duplicates were within their original monthly files, had VendorID 2, and matched on all trip attributes. The Bronze table remains unchanged. The Silver table will retain one row per exact trip signature and exclude the 3 excess copies.

In [ ]:
# Check category values against the expected codes.
from pyspark.sql import functions as F

bronze_df = spark.table("bronze_trips")

code_rules = {
    "VendorID": (
        F.col("VendorID").isNotNull()
        & ~F.col("VendorID").isin(1, 2, 6, 7)
    ),
    "RatecodeID": (
        F.col("RatecodeID").isNotNull()
        & ~F.col("RatecodeID").isin(1, 2, 3, 4, 5, 6, 99)
    ),
    "payment_type": (
        F.col("payment_type").isNotNull()
        & ~F.col("payment_type").isin(0, 1, 2, 3, 4, 5, 6)
    ),
    "store_and_fwd_flag": (
        F.col("store_and_fwd_flag").isNotNull()
        & ~F.col("store_and_fwd_flag").isin("Y", "N")
    )
}

code_expressions = []

for column_name, invalid_condition in code_rules.items():
    # For each field, count every row with an unlisted code and collect each distinct unlisted value once.
    code_expressions.extend([
        F.sum(
            F.when(invalid_condition, 1).otherwise(0)
        ).alias(f"invalid_count__{column_name}"),
        F.collect_set(
            F.when(invalid_condition, F.col(column_name).cast("string"))
        ).alias(f"invalid_values__{column_name}")
    ])

code_profile = bronze_df.agg(*code_expressions).first().asDict()

for column_name in code_rules:
    invalid_count = int(code_profile[f"invalid_count__{column_name}"] or 0)
    invalid_values = sorted(code_profile[f"invalid_values__{column_name}"] or [])
    print(
        f"{column_name}: {invalid_count:,} rows "
        f"| unlisted values: {invalid_values}"
    )

### Unexpected codes
Among non-null values, RatecodeID, payment_type, and store_and_fwd_flag matched the documented code lists. There were 143 rows with VendorID 5, which is not listed in the current TLC data dictionary. These rows remain in Bronze and will be flagged for review; no rows were removed.

In [ ]:
# Compare total_amount with the listed fare and charge components.
from functools import reduce
from pyspark.sql import functions as F

bronze_df = spark.table("bronze_trips")

financial_components = [
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "congestion_surcharge",
    "airport_fee"
]

# Only complete fare components can be compared with the recorded total.
complete_components = reduce(
    lambda left, right: left & right,
    [F.col(column_name).isNotNull() for column_name in financial_components]
) & F.col("total_amount").isNotNull()

component_total = reduce(
    lambda left, right: left + right,
    [F.col(column_name) for column_name in financial_components]
)

# A positive difference means the recorded charge is higher than the sum of its listed components.
difference = F.col("total_amount") - component_total

financial_summary = bronze_df.agg(
    F.sum(F.when(complete_components, 1).otherwise(0))
        .alias("rows_with_complete_components"),
    F.sum(F.when(~complete_components, 1).otherwise(0))
        .alias("rows_excluded_due_to_missing_components"),
    F.sum(
        F.when(complete_components & (F.abs(difference) > 0.01), 1)
         .otherwise(0)
    ).alias("rows_differing_by_more_than_0_01"),
    F.sum(
        F.when(complete_components & (difference > 0.01), 1)
         .otherwise(0)
    ).alias("total_amount_higher_than_components"),
    F.sum(
        F.when(complete_components & (difference < -0.01), 1)
         .otherwise(0)
    ).alias("total_amount_lower_than_components"),
    F.max(
        F.when(complete_components, F.abs(difference))
    ).alias("maximum_absolute_difference")
).first()

for metric_name, metric_value in financial_summary.asDict().items():
    print(f"{metric_name}: {metric_value}")

### Initial check of fare totals
Among 75,288,665 rows with complete financial components, 19,567,921 rows (about 26%) differed from the proposed component sum by more than $0.01. In 19,412,832 rows, total_amount was lower than the component sum; in 155,089 rows, it was higher. The maximum absolute difference was $5.00. This remains a screening result pending investigation; no financial values were changed.

In [ ]:
# Review sample trips to understand amount differences.
from functools import reduce
from pyspark.sql import functions as F

bronze_df = spark.table("bronze_trips")

financial_components = [
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "congestion_surcharge",
    "airport_fee"
]

complete_components = reduce(
    lambda left, right: left & right,
    [F.col(column_name).isNotNull() for column_name in financial_components]
) & F.col("total_amount").isNotNull()

component_total = reduce(
    lambda left, right: left + right,
    [F.col(column_name) for column_name in financial_components]
)

sample_mismatches = (
    bronze_df
    .sample(withReplacement=False, fraction=0.001, seed=42)
    .filter(complete_components)
    .withColumn("component_sum", F.round(component_total, 2))
    .withColumn(
        "difference",
        F.round(F.col("total_amount") - F.col("component_sum"), 2)
    )
    .filter(F.abs(F.col("difference")) > 0.01)
    .select(
        "source_month",
        "VendorID",
        "payment_type",
        "RatecodeID",
        "fare_amount",
        "extra",
        "mta_tax",
        "tip_amount",
        "tolls_amount",
        "improvement_surcharge",
        "congestion_surcharge",
        "airport_fee",
        "total_amount",
        "component_sum",
        "difference"
    )
    .limit(20)
)

sample_mismatches.show(20, truncate=False)

In [ ]:
# Compare amount differences with and without congestion_surcharge.
from pyspark.sql import functions as F

df = spark.table("bronze_trips")

complete = (
    F.col("total_amount").isNotNull()
    & F.col("fare_amount").isNotNull()
    & F.col("extra").isNotNull()
    & F.col("mta_tax").isNotNull()
    & F.col("tip_amount").isNotNull()
    & F.col("tolls_amount").isNotNull()
    & F.col("improvement_surcharge").isNotNull()
    & F.col("congestion_surcharge").isNotNull()
    & F.col("airport_fee").isNotNull()
)

base_sum = (
    F.col("fare_amount") + F.col("extra") + F.col("mta_tax")
    + F.col("tip_amount") + F.col("tolls_amount")
    + F.col("improvement_surcharge") + F.col("airport_fee")
)

# Compare the same complete rows twice to see how including the congestion surcharge changes the mismatch count.
difference_with_surcharge = F.col("total_amount") - (
    base_sum + F.col("congestion_surcharge")
)
difference_without_surcharge = F.col("total_amount") - base_sum

result = df.filter(complete).agg(
    F.count("*").alias("complete_rows"),
    F.sum(F.when(F.abs(difference_with_surcharge) > 0.01, 1).otherwise(0))
        .alias("mismatches_with_surcharge"),
    F.sum(F.when(F.abs(difference_without_surcharge) > 0.01, 1).otherwise(0))
        .alias("mismatches_without_surcharge")
).first()

print(f"Complete rows: {result['complete_rows']:,}")
print(f"Mismatches including surcharge: {result['mismatches_with_surcharge']:,}")
print(f"Mismatches excluding surcharge: {result['mismatches_without_surcharge']:,}")


In [ ]:
# Summarize amount mismatches by congestion_surcharge.
by_surcharge = df.filter(complete).groupBy("congestion_surcharge").agg(
    F.count("*").alias("rows"),
    F.sum(F.when(F.abs(difference_with_surcharge) > 0.01, 1).otherwise(0))
        .alias("mismatches"),
    F.sum(F.when(difference_with_surcharge > 0.01, 1).otherwise(0))
        .alias("total_above_sum"),
    F.sum(F.when(difference_with_surcharge < -0.01, 1).otherwise(0))
        .alias("total_below_sum")
).orderBy("congestion_surcharge")

by_surcharge.show(truncate=False)

In [ ]:
# Break down amount mismatches by vendor and surcharge.
by_vendor = df.filter(complete).groupBy(
    "VendorID", "congestion_surcharge"
).agg(
    F.count("*").alias("rows"),
    F.sum(F.when(F.abs(difference_with_surcharge) > 0.01, 1).otherwise(0))
        .alias("mismatches"),
    F.sum(F.when(difference_with_surcharge < -0.01, 1).otherwise(0))
        .alias("total_below_sum"),
    F.sum(F.when(difference_with_surcharge > 0.01, 1).otherwise(0))
        .alias("total_above_sum")
).orderBy("VendorID", "congestion_surcharge")

by_vendor.show(truncate=False)

### Fare differences by vendor
The reconciliation pattern varied by VendorID. For trips with congestion_surcharge = 2.50, all 18,781,618 VendorID 1 rows had total_amount below the sum of itemized components. For VendorID 2, 145,598 of 50,183,373 rows differed, with 144,196 above the component sum. Another 624,172 VendorID 1 rows with a zero congestion surcharge were below the component sum. This suggests a provider-specific reporting difference, but the cause is not established. Reported financial values are retained; total_amount is not recalculated.

## Silver cleaning rules
- Remove exact duplicate copies.
- Require both timestamps and a drop-off later than pickup.
- Remove pickups outside January 2022 to December 2023.
- Retain other quality exceptions, preserving their original values for review.
- Do not recalculate total_amount or impute missing financial values.

In [ ]:
from pyspark.sql import functions as F

# Read Bronze and keep the starting row count.
bronze_df = spark.table("bronze_trips")
bronze_rows = bronze_df.count()

# Remove exact duplicate rows and keep valid study dates and durations.
trip_columns = [c for c in bronze_df.columns if c not in {"source_file", "source_month"}]
# Match the duplicate profile. If the same trip appears in two files, keep the
# earliest source month and its file name together as one traceable source.
silver_df = (
    bronze_df.groupBy(*trip_columns)
    .agg(F.min(F.struct("source_month", "source_file")).alias("kept_source"))
    .select(*trip_columns, "kept_source.source_file", "kept_source.source_month")
    .select(*bronze_df.columns)
)
pickup_date = F.to_date("tpep_pickup_datetime")
silver_df = silver_df.filter(
    (pickup_date >= F.to_date(F.lit("2022-01-01"))) &
    (pickup_date < F.to_date(F.lit("2024-01-01"))) &
    (F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))
)

# Save the cleaned table and report the row change.
silver_df.write.format("delta").mode("errorifexists").saveAsTable("silver_trips")
silver_rows = spark.table("silver_trips").count()
print(f"Bronze rows: {bronze_rows:,}")
print(f"Silver rows: {silver_rows:,}")
print(f"Rows removed: {bronze_rows - silver_rows:,}")

## Build the date, location and trip tables

In [ ]:
from pyspark.sql import functions as F

# Read the taxi zone lookup for both location dimensions.
zones = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/reference/taxi_zone_lookup.csv")
)

# Build one calendar row for each day in 2022 and 2023.
# The IDs run from 0 to 729. Adding each ID to 1 January 2022 gives one calendar row per day through 31 December 2023.
dim_date = spark.range(730).select(
    F.date_add(
        F.to_date(F.lit("2022-01-01")),
        F.col("id").cast("int")
    ).alias("date")
)
dim_date = dim_date.select(
    F.date_format("date", "yyyyMMdd").cast("int").alias("date_key"),
    "date",
    F.year("date").alias("year"),
    F.month("date").alias("month"),
    F.date_format("date", "MMMM").alias("month_name"),
    F.quarter("date").alias("quarter"),
    F.date_format("date", "EEEE").alias("day_name")
)

# Use the same lookup twice so Power BI can filter either end of a trip independently.
location_columns = [F.col("LocationID").alias("location_id"), "Borough", "Zone", "service_zone"]
dim_pickup_location = zones.select(*location_columns)
dim_dropoff_location = zones.select(*location_columns)

# Save the dimensions as Delta tables.
dim_date.write.format("delta").mode("errorifexists").saveAsTable("dim_date")
dim_pickup_location.write.format("delta").mode("errorifexists").saveAsTable("dim_pickup_location")
dim_dropoff_location.write.format("delta").mode("errorifexists").saveAsTable("dim_dropoff_location")

# Check the number of rows in each dimension.
print(f"DimDate rows: {dim_date.count():,}")
print(f"DimPickupLocation rows: {dim_pickup_location.count():,}")
print(f"DimDropoffLocation rows: {dim_dropoff_location.count():,}")

In [ ]:
from pyspark.sql import functions as F

# Read the cleaned trips.
silver = spark.table("silver_trips")

# Use the pickup date as the calendar key. Each fact row represents one cleaned trip.
fact_trips = silver.select(
    # A pickup on 1 March 2023 gets the key 20230301, which matches that day in dim_date.
    F.date_format(F.to_date("tpep_pickup_datetime"), "yyyyMMdd").cast("int").alias("date_key"),
    F.col("tpep_pickup_datetime").alias("pickup_datetime"),
    F.col("tpep_dropoff_datetime").alias("dropoff_datetime"),
    F.col("PULocationID").alias("pickup_location_id"),
    F.col("DOLocationID").alias("dropoff_location_id"),
    F.col("VendorID").alias("vendor_id"),
    "passenger_count", "trip_distance",
    F.col("RatecodeID").alias("rate_code_id"),
    "payment_type", "fare_amount", "extra", "mta_tax", "tip_amount",
    "tolls_amount", "improvement_surcharge", "congestion_surcharge",
    "airport_fee", "total_amount", "source_month"
)

# Save the one-row-per-trip fact table. Run this write only once.
fact_trips.write.format("delta").mode("errorifexists").saveAsTable("fact_trips")
print("FactTrips table created: fact_trips")
spark.table("fact_trips").show(5, truncate=False)

In [ ]:
from pyspark.sql import functions as F

# Read Silver and define the checks from the cleaning rules.
silver = spark.table("silver_trips")
pickup_date = F.to_date("tpep_pickup_datetime")
invalid_pickup = (
    F.col("tpep_pickup_datetime").isNull() |
    (pickup_date < F.to_date(F.lit("2022-01-01"))) |
    (pickup_date >= F.to_date(F.lit("2024-01-01")))
)
invalid_duration = (
    F.col("tpep_pickup_datetime").isNull() |
    F.col("tpep_dropoff_datetime").isNull() |
    (F.col("tpep_dropoff_datetime") <= F.col("tpep_pickup_datetime"))
)

# Count rows that should have been removed by cleaning.
result = silver.agg(
    F.count("*").alias("silver_rows"),
    F.sum(F.when(invalid_pickup, 1).otherwise(0)).alias("invalid_pickups"),
    F.sum(F.when(invalid_duration, 1).otherwise(0)).alias("invalid_durations")
).first()
print(f"Silver rows: {result['silver_rows']:,}")
print(f"Pickups outside the study period: {result['invalid_pickups']:,}")
print(f"Invalid trip durations: {result['invalid_durations']:,}")

In [ ]:
# Check the stored fact table against Silver before using it in Power BI.
if not result["silver_rows"] or result["invalid_pickups"] or result["invalid_durations"]:
    raise ValueError("Silver failed its row or timestamp checks.")

fact_rows = spark.table("fact_trips").count()
if fact_rows != result["silver_rows"]:
    raise ValueError(f"Fact has {fact_rows:,} rows and Silver has {result['silver_rows']:,}.")
print(f"Fact and Silver both contain {fact_rows:,} rows.")
